# 🧠 MCLDNN — Baseline Training & Evaluation

**Model:** Multi-Channel LSTM Deep Neural Network (MCLDNN)

**Dataset:** RadioML 2016.10a (11 modulations, 220K samples, 2×128 IQ)

**Reference:** Xu et al., "A Spatiotemporal Multi-Channel Learning Framework for AMR", IEEE 2020

---

Bu notebook, MCLDNN modelinin orijinal AWGN koşullarında baseline eğitimini ve değerlendirmesini gerçekleştirir.

**Adımlar:**
1. Ortam kurulumu
2. Dataset yükleme
3. Model oluşturma
4. Eğitim
5. Değerlendirme (Confusion Matrix, SNR vs Accuracy)

## 1. Ortam Kurulumu

In [ ]:
# GPU kontrolü
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Gerekli kütüphaneler
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

print("\n✅ Ortam hazır.")

In [ ]:
# Google Drive bağlantısı
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Projeyi klonla (ilk çalıştırmada)
REPO_URL = 'https://github.com/YOUR_USERNAME/AMR-UnderDifferentNoises-DL.git'  # <-- BURAYI GÜNCELLEYIN
PROJECT_DIR = '/content/AMR-UnderDifferentNoises-DL'

if not os.path.exists(PROJECT_DIR):
    !git clone -b dev/project-restructure {REPO_URL}
else:
    print(f"Proje zaten mevcut: {PROJECT_DIR}")

# src/ modüllerini import edebilmek için path ekle
import sys
sys.path.insert(0, PROJECT_DIR)

print(f"\n✅ Proje dizini: {PROJECT_DIR}")

## 2. Dataset Yükleme

RML2016.10a dataset dosyasını Google Drive'ınıza yükleyin:

📂 `MyDrive/AMR-Project/RML2016.10a_dict.pkl`

In [ ]:
# Dataset path ayarları
# Öncelik: Drive > Lokal
DRIVE_DATASET = '/content/drive/MyDrive/AMR-Project/RML2016.10a_dict.pkl'
LOCAL_DATASET = os.path.join(PROJECT_DIR, 'data', 'RML2016.10a_dict.pkl')

if os.path.exists(DRIVE_DATASET):
    DATASET_PATH = DRIVE_DATASET
elif os.path.exists(LOCAL_DATASET):
    DATASET_PATH = LOCAL_DATASET
else:
    raise FileNotFoundError(
        f"Dataset bulunamadı!\n"
        f"  Drive: {DRIVE_DATASET}\n"
        f"  Lokal: {LOCAL_DATASET}\n"
        f"Lütfen RML2016.10a_dict.pkl dosyasını yukarıdaki konumlardan birine yerleştirin."
    )

print(f"📁 Dataset: {DATASET_PATH}")
print(f"📊 Boyut: {os.path.getsize(DATASET_PATH) / 1024 / 1024:.1f} MB")

In [ ]:
# Veri yükleme
from src.utils.dataset import load_data, prepare_data_mcldnn

(mods, snrs, lbl), (X_train, Y_train), (X_val, Y_val), \
    (X_test, Y_test), (train_idx, val_idx, test_idx) = load_data(DATASET_PATH)

print(f"\n📋 Modülasyonlar ({len(mods)}): {mods}")
print(f"📡 SNR aralığı: {snrs[0]} → {snrs[-1]} dB ({len(snrs)} seviye)")
print(f"\n🔢 Veri boyutları:")
print(f"   Train: X={X_train.shape}, Y={Y_train.shape}")
print(f"   Val:   X={X_val.shape},   Y={Y_val.shape}")
print(f"   Test:  X={X_test.shape},  Y={Y_test.shape}")

In [ ]:
# MCLDNN formatına dönüştürme
data = prepare_data_mcldnn(X_train, X_val, X_test)

print("MCLDNN Input Formatları:")
print(f"  input1 (IQ combined): {data['train'][0].shape}")  # (N, 2, 128, 1)
print(f"  input2 (I channel):   {data['train'][1].shape}")  # (N, 128, 1)
print(f"  input3 (Q channel):   {data['train'][2].shape}")  # (N, 128, 1)

## 3. Veri Görselleştirme

In [ ]:
# Örnek IQ sinyali görselleştirme
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
fig.suptitle('Örnek IQ Sinyalleri (Farklı Modülasyonlar)', fontsize=14)

sample_mods = ['BPSK', 'QPSK', '8PSK', '16-QAM', '64-QAM', 'WBFM']
for idx, mod_name in enumerate(sample_mods):
    ax = axes[idx // 3, idx % 3]
    # İlk SNR=10dB örneğini bul
    for i, (m, s) in enumerate([(lbl[j][0], lbl[j][1]) for j in train_idx[:1000]]):
        if m == mod_name and s == 10:
            sample = X_train[i]
            ax.plot(sample[0], label='I', alpha=0.8, linewidth=0.8)
            ax.plot(sample[1], label='Q', alpha=0.8, linewidth=0.8)
            ax.set_title(f'{mod_name} (SNR=10dB)', fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            break

plt.tight_layout()
plt.show()

## 4. Model Oluşturma

In [ ]:
from src.models.mcldnn import MCLDNN

model = MCLDNN(weights=None, classes=len(mods))
model.compile(
    loss='categorical_crossentropy',
    metrics=['accuracy'],
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)
)

model.summary()

## 5. Eğitim

In [ ]:
# Sonuçların kaydedileceği dizin
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', 'mcldnn', 'awgn')
os.makedirs(os.path.join(RESULTS_DIR, 'weights'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'predictions'), exist_ok=True)

WEIGHTS_PATH = os.path.join(RESULTS_DIR, 'weights', 'best_weights.h5')

# Eğitim parametreleri
EPOCHS = 1000
BATCH_SIZE = 400

print(f"📂 Sonuçlar: {RESULTS_DIR}")
print(f"⚖️  Weights: {WEIGHTS_PATH}")
print(f"🔄 Epochs: {EPOCHS}, Batch: {BATCH_SIZE}")

In [ ]:
# Callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        WEIGHTS_PATH, monitor='val_loss',
        verbose=1, save_best_only=True, mode='auto'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        verbose=1, patience=5, min_lr=1e-7
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=50,
        verbose=1, mode='auto', restore_best_weights=True
    )
]

# Eğitim başlat
history = model.fit(
    data['train'], Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=2,
    validation_data=(data['val'], Y_val),
    callbacks=callbacks
)

print("\n✅ Eğitim tamamlandı!")

In [ ]:
# Eğitim grafiklerini kaydet
from src.utils.metrics import plot_training_history

plot_training_history(history, save_dir=os.path.join(RESULTS_DIR, 'figures'))

# Eğitim grafiklerini göster
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'], label='Train')
ax1.plot(history.history['val_loss'], label='Val')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

acc_key = 'accuracy' if 'accuracy' in history.history else 'acc'
val_acc_key = 'val_accuracy' if 'val_accuracy' in history.history else 'val_acc'
ax2.plot(history.history[acc_key], label='Train')
ax2.plot(history.history[val_acc_key], label='Val')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Değerlendirme

In [ ]:
# En iyi ağırlıkları yükle
model.load_weights(WEIGHTS_PATH)
print(f"✅ Ağırlıklar yüklendi: {WEIGHTS_PATH}")

In [ ]:
# Tam değerlendirme pipeline'ı
from src.utils.metrics import evaluate_model

acc, acc_mod_snr = evaluate_model(
    model=model,
    X_test_inputs=data['test'],
    Y_test=Y_test,
    lbl=lbl,
    test_idx=test_idx,
    snrs=snrs,
    classes=mods,
    batch_size=BATCH_SIZE,
    results_dir=RESULTS_DIR
)

In [ ]:
# SNR vs Accuracy grafiğini göster
plt.figure(figsize=(10, 6))
plt.plot(snrs, [acc[s] for s in snrs], 'o-', linewidth=2, markersize=6, color='#2196F3')
plt.xlabel('SNR (dB)', fontsize=12)
plt.ylabel('Classification Accuracy', fontsize=12)
plt.title('MCLDNN — SNR vs Classification Accuracy (Baseline AWGN)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.ylim([0, 1])
plt.tight_layout()
plt.show()

# Özet
print(f"\n📊 MCLDNN Baseline Sonuçları (AWGN):")
print(f"   Ortalama accuracy: {np.mean(list(acc.values()))*100:.2f}%")
print(f"   En düşük (SNR={snrs[0]}dB): {acc[snrs[0]]*100:.2f}%")
print(f"   En yüksek (SNR={snrs[-1]}dB): {acc[snrs[-1]]*100:.2f}%")

In [ ]:
# Sonuçları Drive'a kaydet
DRIVE_RESULTS = '/content/drive/MyDrive/AMR-Project/results/mcldnn_awgn'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Ağırlıkları kopyala
import shutil
shutil.copy2(WEIGHTS_PATH, os.path.join(DRIVE_RESULTS, 'best_weights.h5'))

# Sonuçları kaydet
with open(os.path.join(DRIVE_RESULTS, 'acc.pkl'), 'wb') as f:
    pickle.dump(acc, f)
with open(os.path.join(DRIVE_RESULTS, 'acc_mod_snr.pkl'), 'wb') as f:
    pickle.dump(acc_mod_snr, f)

print(f"\n💾 Sonuçlar kaydedildi: {DRIVE_RESULTS}")

---
## ✅ Tamamlandı

MCLDNN baseline eğitimi ve değerlendirmesi tamamlandı.

**Sonraki adım:** `02_baseline_petcgdnn.ipynb` notebook'unu çalıştırın.